# EWS Feature Engineering

This notebook creates a ML feature table for the Early Warning System.

The previous preprocessing notebook produced a clean baseline modeling dataset. This notebook enriches it with business-relevant and leakage-safe features from product, inventory tables.

Main steps:

1. Load the cleaned product-month dataset and raw supporting tables.
2. Join product metadata, inventory details.
3. Create ML features:
   - product age and lifecycle features
   - margin and gross-profit features
   - campaign spend and digital demand features
   - lead-time and stockout pressure features
   - seasonality and macro-market features
   - category-region benchmark features
   - relative product performance features
   - rolling momentum, acceleration, and stability features
4. Avoid target leakage:
   - no future target columns are used as features
   - rolling benchmark features use historical or current-month observable data only
   - the final month with missing future label is kept for scoring but excluded from train/validation/test
5. Export:
   - a human-readable feature-engineered dataset
   - train/validation/test ML-ready matrices
   - ID files for traceability
   - a compact feature engineering quality report

## Step Overview

Before the code runs, here is the logic in plain language:

- We start from the cleaned baseline table because it already contains safe lag, rolling, and target fields.
- We enrich it with product metadata, inventory lead times, campaign spend, product page views, seasonality, and macro-healthcare context.
- We create stronger features that explain *why* a product may be risky, not only whether sales dropped.
- We fit imputation, scaling, and one-hot encoding only on the training period, then apply the same transformations to validation and test.
- We keep IDs separately so predictions can be traced back to product, month, and region.


In [291]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42

INPUT_DIR = Path("../data/preprocessed")
INPUT_DIR_MARKET_DATA = Path("../data/new_generated_data")
OUTPUT_DIR = Path("../data/feature_engineering")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "next_month_risk_label"
ID_COLS = ["product_id", "year_month", "region_id"]

In [292]:
# Divide safely and return NaN where denominator is zero or missing
def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)

# Linear trend slope for a rolling window
def slope(values):
    values = np.asarray(values, dtype=float)
    if len(values) < 2 or np.all(np.isnan(values)):
        return np.nan
    x = np.arange(len(values))
    return float(np.polyfit(x, values, 1)[0])

In [ ]:
# Load cleaned modeling data plus supporting raw business tables
def load_inputs():
    paths = {
        "modeling": INPUT_DIR / "final_modeling_dataset_cleaned.csv",
        "products": INPUT_DIR / "products.csv",
        "inventory": INPUT_DIR / "inventory.csv",
        "marketing": INPUT_DIR_MARKET_DATA / "factMarketActivity.csv",
        "market": INPUT_DIR_MARKET_DATA / "factMarketSignals.csv",
        "transactions": INPUT_DIR / "sales_transactions.csv",
        "region": INPUT_DIR / "dimRegion.csv",
        "customers": INPUT_DIR / "customers.csv"}

    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing input files:\n" + "\n".join(missing))

    data = {name: pd.read_csv(path) for name, path in paths.items()}
    data["products"]["launch_date"] = pd.to_datetime(data["products"]["launch_date"])
    data["transactions"]["order_date"] = pd.to_datetime(data["transactions"]["order_date"])
    region = data["region"][["region", "region_id"]]
    data["modeling"] = data["modeling"].merge(region, on ="region", how="left")

    return data

In [ ]:
# Join source tables that contain useful ML signals not fully included in baseline preprocessing
def enrich_with_source_tables(data):
    df = data["modeling"].copy()

    products = data["products"][[
        "product_id",
        "product_name",
        "product_group"]].copy()

    inventory = data["inventory"][
        ["product_id", "year_month", "region_id"]].copy()

    marketing = data["marketing"][
        ["product_id", "year_month", "region_id", "campaign_spend_eur", "product_page_views"]].copy()

    market = data["market"][[
        "product_group",
        "year_month",
        "region_id",
        "seasonality_index",
        "macro_business_index"]
    ].copy()

    df = df.merge(products, on="product_id", how="left")
    df = df.merge(inventory, on=["product_id", "year_month", "region_id"], how="left")
    df = df.merge(marketing, on=["product_id", "year_month", "region_id"], how="left")
    df = df.merge(market, on=["product_group", "year_month", "region_id"], how="left")

    return df.sort_values(["product_id", "region_id", "year_month"]).reset_index(drop=True)

# Add product maturity, lifecycle, strategic, and margin features
def add_product_lifecycle_features(df):
    out = df.copy()

    out["year_month"] = (
        pd.to_datetime(out["year_month"].astype(str), errors="raise")
        .dt.to_period("M")
        .dt.to_timestamp())

    out["launch_date"] = pd.to_datetime(
        out["launch_date"],
        errors="raise")
    
    # out["is_strategic_product"] = out["is_strategic_product"].astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)
    out["product_age_months"] = (
        (out["year_month"].dt.year - out["launch_date"].dt.year) * 12
        + (out["year_month"].dt.month - out["launch_date"].dt.month)
    ).clip(lower=0)

    out["is_new_product"] = (out["product_age_months"] <= 12).astype(int)
    # out["is_mature_product"] = (out["product_age_months"] >= 36).astype(int)

    lifecycle_order = {
        "New": 0,
        "Growth": 1,
        "Mature": 2,
        "Decline": 3}
    out["lifecycle_stage_num"] = out["lifecycle_stage"].map(lifecycle_order).fillna(-1)
    out["is_decline_stage"] = (out["lifecycle_stage"] == "Decline").astype(int)
    out["is_growth_stage"] = (out["lifecycle_stage"] == "Growth").astype(int)

    out["estimated_gross_profit"] = out["revenue"] - out["units_sold"] * out["base_unit_cost_eur"]
    out["estimated_gross_margin_pct"] = safe_divide(out["estimated_gross_profit"], out["revenue"])
    out["margin_gap_vs_target"] = out["estimated_gross_margin_pct"] - out["target_margin_pct"]
    # out["strategic_low_margin_flag"] = ((out["is_strategic_product"] == 1) & (out["margin_gap_vs_target"] < -0.10)).astype(int)

    return out

# Add campaign intensity and demand-funnel features
def add_marketing_and_demand_features(df):
    out = df.copy()

    out["campaign_spend_eur"] = pd.to_numeric(out["campaign_spend_eur"], errors="coerce").fillna(0)
    out["product_page_views"] = pd.to_numeric(out["product_page_views"], errors="coerce").fillna(0)
    out["website_visits"] = pd.to_numeric(out["website_visits"], errors="coerce").fillna(0)
    out["demo_requests"] = pd.to_numeric(out["demo_requests"], errors="coerce").fillna(0)

    out["campaign_spend_per_visit"] = safe_divide(out["campaign_spend_eur"], out["website_visits"])
    out["campaign_spend_per_demo"] = safe_divide(out["campaign_spend_eur"], out["demo_requests"])
    out["page_view_to_visit_ratio"] = safe_divide(out["product_page_views"], out["website_visits"])
    out["demo_conversion_rate"] = safe_divide(out["demo_requests"], out["product_page_views"])
    out["digital_interest_per_unit"] = safe_divide(out["product_page_views"], out["units_sold"])
    out["campaign_active_with_no_sales"] = ((out["campaign_spend_eur"] > 0) & (out["units_sold"] == 0)).astype(int)

    group = out.groupby(["product_id", "region_id"], group_keys=False)
    out["campaign_spend_lag_1m"] = group["campaign_spend_eur"].shift(1)
    out["campaign_spend_rolling_3m"] = group["campaign_spend_eur"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())
    out["website_visits_growth_pct"] = group["website_visits"].pct_change()
    out["demo_requests_growth_pct"] = group["demo_requests"].pct_change()

    return out

# Add seasonality, macro, and competitive context features
def add_market_context_features(df):
    out = df.copy()

    out["seasonality_index"] = pd.to_numeric(out["seasonality_index"], errors="coerce")
    out["macro_business_index"] = pd.to_numeric(out["macro_business_index"], errors="coerce")

    out["seasonality_adjusted_units"] = safe_divide(out["units_sold"], out["seasonality_index"] / 100)
    out["demand_competition_ratio"] = safe_divide(out["market_demand_index"], out["competitor_pressure_index"])
    out["market_tailwind_score"] = (
        (out["market_demand_index"].fillna(100) - 100) * 0.50
        + (100 - out["competitor_pressure_index"].fillna(50)) * 0.30
        + (out["macro_business_index"].fillna(100) - 100) * 0.20)
    out["high_competition_flag"] = (out["competitor_pressure_index"] >= 70).astype(int)
    out["weak_market_flag"] = (out["market_demand_index"] < 90).astype(int)

    group = out.groupby(["product_group", "region_id"], group_keys=False)
    out["market_demand_change_pct"] = group["market_demand_index"].pct_change()
    out["macro_business_index_pct"] = group["macro_business_index"].pct_change()
    out["competitor_pressure_change_pct"] = group["competitor_pressure_index"].pct_change()

    return out

# Compare each product to category-region peers and its own recent history
def add_relative_performance_features(df):
    out = df.copy()

    category_region = out.groupby(["product_group", "region_id", "year_month"], as_index=False).agg(
    category_region_units_2=("units_sold", "sum"),
    category_region_revenue=("revenue", "sum"),
    category_region_customers=("unique_customers", "sum"),
    category_region_avg_discount=("avg_discount_pct", "mean"))

    out = out.merge(category_region, on=["product_group", "region_id", "year_month"], how="left")

    out["product_unit_share_category_region"] = safe_divide(out["units_sold"], out["category_region_units_2"])
    out["product_revenue_share_category_region"] = safe_divide(out["revenue"], out["category_region_revenue"])
    out["customer_share_category_region"] = safe_divide(out["unique_customers"], out["category_region_customers"])
    out["discount_vs_category_region"] = out["avg_discount_pct"] - out["category_region_avg_discount"]

    group_pr = out.groupby(["product_id", "region"], group_keys=False)
    out["unit_share_lag_1m"] = group_pr["product_unit_share_category_region"].shift(1)
    out["unit_share_change_pct"] = safe_divide(
        out["product_unit_share_category_region"] - out["unit_share_lag_1m"],
        out["unit_share_lag_1m"])

    out["units_acceleration_3m"] = group_pr["trend_slope_3m"].diff()
    out["units_vs_6m_avg_pct"] = safe_divide(out["units_sold"] - out["rolling_units_mean_6m"], out["rolling_units_mean_6m"])
    out["revenue_vs_lag_1m_pct"] = safe_divide(out["revenue"] - out["revenue_lag_1m"], out["revenue_lag_1m"])

    out["rolling_units_cv_6m"] = group_pr["units_sold"].transform(
        lambda s: safe_divide(
            s.shift(1).rolling(6, min_periods=3).std(),
            s.shift(1).rolling(6, min_periods=3).mean()))

    out["rolling_revenue_mean_3m"] = group_pr["revenue"].transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
    out["rolling_revenue_std_3m"] = group_pr["revenue"].transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
    out["revenue_volatility_3m"] = safe_divide(out["rolling_revenue_std_3m"], out["rolling_revenue_mean_3m"])

    return out

# Create customer-mix features from transactional data without using future months
def add_customer_mix_features(df, transactions, customers):
    tx = transactions.merge(
        customers[["customer_id", "customer_segment", "customer_size_score", "base_churn_probability"]],
        on="customer_id",
        how="left")

    tx["year_month"] = pd.to_datetime(tx["year_month"],errors="raise")

    # Beide Merge-Spalten identisch als Monatsanfang formatieren
    df["year_month"] = (
        pd.to_datetime(
            df["year_month"].astype(str),
            errors="raise")
        .dt.to_period("M")
        .dt.to_timestamp())

    tx["year_month"] = (
        pd.to_datetime(
            tx["year_month"].astype(str),
            errors="raise")
        .dt.to_period("M")
        .dt.to_timestamp())
        
    tx["is_distributor"] = (tx["customer_segment"] == "Distributor").astype(int)

    # Ab 12 % Basiswahrscheinlichkeit als erhöhtes Risiko klassifizieren
    tx["is_churn_risk_customer"] = (tx["base_churn_probability"] >= 0.12).astype(int)

    mix = tx.groupby(["product_id", "region_id", "year_month"], as_index=False).agg(
        avg_customer_size_score=("customer_size_score", "mean"),
        distributor_order_share=("is_distributor", "mean"),
        avg_customer_churn_probability=("base_churn_probability", "mean"),
        churn_risk_customer_share=("is_churn_risk_customer", "mean"),
        avg_order_units=("units_sold", "mean"),
        avg_order_revenue=("revenue", "mean"))

    out = df.merge(mix, on=["product_id", "region_id", "year_month"], how="left")

    group = out.groupby(["product_id", "region_id"], group_keys=False)
    out["avg_customer_size_score_lag_1m"] = group["avg_customer_size_score"].shift(1)
    out["avg_customer_churn_probability_lag_1m"] = group["avg_customer_churn_probability"].shift(1)
    out["distributor_order_share_lag_1m"] = group["distributor_order_share"].shift(1)
    out["churn_risk_customer_share_lag_1m"] = group["churn_risk_customer_share"].shift(1)

    return out

In [295]:
# Create a compact catalog of engineered features for documentation
def create_feature_catalog(df):
    engineered_features = [
        col for col in df.columns
        if col not in ID_COLS
        and col not in ["product_name", "launch_date", TARGET_COL]]

    catalog_rows = []
    for col in engineered_features:
        if col in ["product_line", "product_category", "region", "lifecycle_stage"]:
            feature_type = "categorical"
        elif df[col].dropna().isin([0, 1]).all():
            feature_type = "binary"
        else:
            feature_type = "numeric"

        if any(token in col for token in ["future", "next_month"]):
            leakage_status = "exclude"
        else:
            leakage_status = "ok"

        catalog_rows.append({
            "feature": col,
            "feature_type": feature_type,
            "missing_rate": round(float(df[col].isna().mean()), 4),
            "leakage_status": leakage_status})

    return pd.DataFrame(catalog_rows).sort_values(["leakage_status", "feature"]).reset_index(drop=True)

# Return categorical and numeric feature lists for ML-ready export
def get_model_feature_lists(df):
    exclude = set(ID_COLS + [TARGET_COL, "product_name", "launch_date"])
    leakage_like = {c for c in df.columns if c.startswith("future_") or (("next_month" in c) and c != TARGET_COL)}
    exclude = leakage_like

    categorical_features = [
        col for col in ["product_line", "product_category", "region", "lifecycle_stage"]
        if col in df.columns]

    numeric_features = [
        col for col in df.columns
        if col not in exclude
        and col not in categorical_features
        and pd.api.types.is_numeric_dtype(df[col])]

    return numeric_features, categorical_features

# Fit median imputation, standardization, and categorical level lists on train only
def fit_pandas_preprocessor(train, numeric_features, categorical_features):
    numeric_medians = train[numeric_features].median(numeric_only=True).fillna(0)
    filled = train[numeric_features].fillna(numeric_medians)
    numeric_means = filled.mean()
    numeric_stds = filled.std().replace(0, 1).fillna(1)

    categorical_modes = {}
    categorical_levels = {}
    for col in categorical_features:
        mode = train[col].mode(dropna=True)
        categorical_modes[col] = mode.iloc[0] if len(mode) else "Unknown"
        categorical_levels[col] = sorted(train[col].fillna(categorical_modes[col]).astype(str).unique().tolist())

    return {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "numeric_medians": numeric_medians,
        "numeric_means": numeric_means,
        "numeric_stds": numeric_stds,
        "categorical_modes": categorical_modes,
        "categorical_levels": categorical_levels}

# Transform a dataframe into a numeric ML matrix
def transform_with_pandas_preprocessor(df, preprocessor):
    numeric_features = preprocessor["numeric_features"]
    categorical_features = preprocessor["categorical_features"]

    numeric = df[numeric_features].copy()
    numeric = numeric.fillna(preprocessor["numeric_medians"])
    numeric = (numeric - preprocessor["numeric_means"]) / preprocessor["numeric_stds"]
    parts = [numeric.reset_index(drop=True)]

    for col in categorical_features:
        values = df[col].fillna(preprocessor["categorical_modes"][col]).astype(str)
        levels = preprocessor["categorical_levels"][col]
        encoded = pd.DataFrame(
            {f"{col}__{level}": (values == level).astype(int).to_numpy() for level in levels})
        parts.append(encoded)
    return pd.concat(parts, axis=1)

# Create chronological train/validation/test splits
def create_time_splits(df):
    labeled = df.dropna(subset=[TARGET_COL]).copy()
    labeled[TARGET_COL] = labeled[TARGET_COL].astype(int)

    months = sorted(labeled["year_month"].unique())
    train_end = months[int(len(months) * 0.70)]
    valid_end = months[int(len(months) * 0.85)]

    train = labeled[labeled["year_month"] <= train_end].copy()
    valid = labeled[(labeled["year_month"] > train_end) & (labeled["year_month"] <= valid_end)].copy()
    test = labeled[labeled["year_month"] > valid_end].copy()

    return train, valid, test

# Export feature-engineered ML-ready train/validation/test files
def export_ml_ready(train, valid, test, numeric_features, categorical_features):
    preprocessor = fit_pandas_preprocessor(train, numeric_features, categorical_features)

    train_ml = transform_with_pandas_preprocessor(train, preprocessor)
    valid_ml = transform_with_pandas_preprocessor(valid, preprocessor)
    test_ml = transform_with_pandas_preprocessor(test, preprocessor)

    train_ml[TARGET_COL] = train[TARGET_COL].to_numpy()
    valid_ml[TARGET_COL] = valid[TARGET_COL].to_numpy()
    test_ml[TARGET_COL] = test[TARGET_COL].to_numpy()

    train_ml.to_csv(OUTPUT_DIR / "train_feature_engineered_ml_ready.csv", index=False)
    valid_ml.to_csv(OUTPUT_DIR / "valid_feature_engineered_ml_ready.csv", index=False)
    test_ml.to_csv(OUTPUT_DIR / "test_feature_engineered_ml_ready.csv", index=False)

    train[ID_COLS + [TARGET_COL]].to_csv(OUTPUT_DIR / "train_feature_engineered_ids.csv", index=False)
    valid[ID_COLS + [TARGET_COL]].to_csv(OUTPUT_DIR / "valid_feature_engineered_ids.csv", index=False)
    test[ID_COLS + [TARGET_COL]].to_csv(OUTPUT_DIR / "test_feature_engineered_ids.csv", index=False)

    return {
        "train_shape": train_ml.shape,
        "valid_shape": valid_ml.shape,
        "test_shape": test_ml.shape,
        "feature_count": train_ml.shape[1] - 1,
        "feature_names": [c for c in train_ml.columns if c != TARGET_COL]}

In [298]:
def run_feature_engineering():
    print("Loading inputs...")
    data = load_inputs()

    print("Joining supporting business tables...")
    df = enrich_with_source_tables(data)

    print("Adding product lifecycle and margin features...")
    df = add_product_lifecycle_features(df)

    print("Adding marketing and digital-demand features...")
    df = add_marketing_and_demand_features(df)

#    print("Adding supply-chain pressure features...")
#    df = add_supply_chain_features(df)

    print("Adding market-context features...")
    df = add_market_context_features(df)

    print("Adding category-region relative performance features...")
    df = add_relative_performance_features(df)

    print("Adding customer-mix features...")
    df = add_customer_mix_features(df, data["transactions"], data["customers"])

    # Clean obvious infinite values caused by zero denominators
    # Keep NaNs for train-only imputation
    df = df.replace([np.inf, -np.inf], np.nan)

    feature_catalog = create_feature_catalog(df)
    numeric_features, categorical_features = get_model_feature_lists(df)

    print("Creating chronological train/validation/test splits...")
    train, valid, test = create_time_splits(df)

    print("Exporting feature-engineered datasets...")
    ml_summary = export_ml_ready(train, valid, test, numeric_features, categorical_features)

    df.to_csv(OUTPUT_DIR / "feature_engineered_modeling_dataset.csv", index=False)
    feature_catalog.to_csv(OUTPUT_DIR / "feature_catalog.csv", index=False)

    report = {
        "feature_engineered_dataset_shape": list(df.shape),
        "numeric_feature_count": len(numeric_features),
        "categorical_feature_count": len(categorical_features),
        "ml_feature_count_after_encoding": ml_summary["feature_count"],
        "train_shape": list(ml_summary["train_shape"]),
        "valid_shape": list(ml_summary["valid_shape"]),
        "test_shape": list(ml_summary["test_shape"]),
        "target_distribution": df[TARGET_COL].value_counts(dropna=False).sort_index().astype(int).to_dict(),
        "top_missing_rates": (
            df[numeric_features + categorical_features + [TARGET_COL]]
            .isna()
            .mean()
            .sort_values(ascending=False)
            .head(20)
            .round(4)
            .to_dict()),
        "leakage_like_feature_columns": [
            c for c in numeric_features + categorical_features
            if c.startswith("future_") or (("next_month" in c) and c != TARGET_COL)]}

    with open(OUTPUT_DIR / "feature_engineering_quality_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, default=str)

    print("\nFeature engineering complete")
    print("-" * 80)
    print(f"Feature-engineered dataset shape: {df.shape}")
    print(f"Numeric features before encoding: {len(numeric_features)}")
    print(f"Categorical features before encoding: {len(categorical_features)}")
    print(f"ML features after encoding: {ml_summary['feature_count']}")
    print(f"Train ML shape: {ml_summary['train_shape']}")
    print(f"Validation ML shape: {ml_summary['valid_shape']}")
    print(f"Test ML shape: {ml_summary['test_shape']}")
    print("\nTarget distribution:")
    print(df[TARGET_COL].value_counts(dropna=False).sort_index().to_string())
    print("\nLeakage-like feature columns used:")
    print(report["leakage_like_feature_columns"] if report["leakage_like_feature_columns"] else "none")
    print("\nSample engineered columns:")
    sample_cols = [
        "product_id", "region", "product_age_months", "lifecycle_stage",
        "estimated_gross_margin_pct", "campaign_spend_rolling_3m", "market_tailwind_score", "product_unit_share_category_region", TARGET_COL]
    print(df[sample_cols].sample(8, random_state=RANDOM_SEED).to_string(index=False))
    print("\nOutput files saved to:", OUTPUT_DIR.resolve())

    return {
        "feature_engineered": df,
        "feature_catalog": feature_catalog,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "report": report}

artifacts = run_feature_engineering()


Loading inputs...
Joining supporting business tables...
Adding product lifecycle and margin features...
Adding marketing and digital-demand features...
Adding market-context features...
Adding category-region relative performance features...
Adding customer-mix features...
Creating chronological train/validation/test splits...
Exporting feature-engineered datasets...

Feature engineering complete
--------------------------------------------------------------------------------
Feature-engineered dataset shape: (98520, 148)
Numeric features before encoding: 136
Categorical features before encoding: 4
ML features after encoding: 164
Train ML shape: (62520, 165)
Validation ML shape: (18000, 165)
Test ML shape: (16000, 165)

Target distribution:
next_month_risk_label
0.0    12519
1.0    50711
2.0    33290
NaN     2000

Leakage-like feature columns used:
none

Sample engineered columns:
 product_id          region  product_age_months lifecycle_stage  estimated_gross_margin_pct  campaign_spen